In [1]:
from julia.api import Julia
from julia import Main

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import os

import pathlib
from pathlib import Path
import shutil

Which Fit?

In [2]:
fit_name = "MSHT20N3LO-MC"
if_grid = True
if_generate_table = False # True: generate table; False: generate and store predictions
integrator = "MultiGKTable" # "MultiGKTable", "quadgk"
N_cores = 16

Main.eval('using Distributed')
Main.eval(f'addprocs({N_cores})')

array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17],
      dtype=int64)

Read Files

In [3]:
Main.if_grid = if_grid
Main.fit_name = fit_name
Main.if_generate_table = if_generate_table
Main.integrator = integrator

Main.eval("""
using Distributed
@everywhere begin
    # declare globals in this process's Main
    global if_grid, fit_name, if_generate_table

    # copy values from pid 1 explicitly
    if_grid           = remotecall_fetch(() -> Main.if_grid, 1)
    fit_name          = remotecall_fetch(() -> Main.fit_name, 1)
    if_generate_table = remotecall_fetch(() -> Main.if_generate_table, 1)
    integrator = remotecall_fetch(() -> Main.integrator, 1)          

end
""")

TMD_fitting_root = "../"
def include(name):
    path = os.path.join(TMD_fitting_root, name)
    Main.eval(f'@everywhere include(raw"{path}")')

include(f"Cards/{fit_name}.jl")

Main.eval(r'''
@everywhere begin
    for (i, name) in enumerate(param_names)
        s = Symbol("NP_", name)
        Core.eval(Main, :(const $(s) = Float32(initial_params[$i])))
    end
end
''')

# Grids
if if_grid == True:
    include("Grids/initialization.jl")
# PDF
include("Collinear_PDF/pdf.jl")
# Core
include("Core/constants.jl")
include("Core/anomalous dims.jl")
include("Core/strong coupling.jl")
# Numerical
include("Numerical/FastGK.jl")
include("Numerical/MultiGKTable.jl")
include("Numerical/AdaptiveGKNodes.jl")
include("Numerical/budget_QuadGK.jl")

# TMDs
include("TMDs/anomalous dims.jl")
include(f"TMDs/TMDPDFs/{Main.TMDPDF_name}")
include(f"TMDs/NP Parameterizations/{Main.NP_name}")
# DY
include(f"DY/DY_table.jl")

# Data
file_root = f"../Data/{Main.data_name}/Cutted/DY"
table_root = f"../Tables/{Main.table_name}/DY"
total_root = f"../Data/DY_total_xsec/{Main.total_xsec_name}"

#Main.eval('@everywhere @show myid(), flavor_scheme')

LHAPDF 6.5.5 loading /home/maxzhang/UCLA_QCD_fitter/installation/share/LHAPDF/MSHT20an3lo_as118_mc/MSHT20an3lo_as118_mc_0000.dat
MSHT20an3lo_as118_mc PDF set, member #0, version 1
      From worker 4:	LHAPDF 6.5.5 loading /home/maxzhang/UCLA_QCD_fitter/installation/share/LHAPDF/MSHT20an3lo_as118_mc/MSHT20an3lo_as118_mc_0000.dat
      From worker 4:	MSHT20an3lo_as118_mc PDF set, member #0, version 1
      From worker 12:	LHAPDF 6.5.5 loading /home/maxzhang/UCLA_QCD_fitter/installation/share/LHAPDF/MSHT20an3lo_as118_mc/MSHT20an3lo_as118_mc_0000.dat
      From worker 12:	MSHT20an3lo_as118_mc PDF set, member #0, version 1
      From worker 7:	LHAPDF 6.5.5 loading /home/maxzhang/UCLA_QCD_fitter/installation/share/LHAPDF/MSHT20an3lo_as118_mc/MSHT20an3lo_as118_mc_0000.dat
      From worker 7:	MSHT20an3lo_as118_mc PDF set, member #0, version 1
      From worker 5:	LHAPDF 6.5.5 loading /home/maxzhang/UCLA_QCD_fitter/installation/share/LHAPDF/MSHT20an3lo_as118_mc/MSHT20an3lo_as118_mc_0000.dat
  

In [4]:
def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df

df_predictions = dict([])
df_predictions_ogata = dict([])

By file or by experiment?

In [5]:
data_selections = "by_experiment"  # "by_file" or "by_experiment"

In [6]:
experiments =[
    'ATLAS_7',
    'ATLAS_8',
    #'ATLAS_13',
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]

In [7]:
from pathlib import Path

if data_selections == "by_experiment":
    file_names = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        for p in sorted(exp_dir.glob("*.csv")):
            file_names.append(str(Path(experiment) / p.name))  # e.g. "E605/E605_Q_10.5_11.5.csv")

display(file_names)

['ATLAS_7/ATLAS7-00y10.csv',
 'ATLAS_7/ATLAS7-10y20.csv',
 'ATLAS_7/ATLAS7-20y24.csv',
 'ATLAS_8/ATLAS8-00y04.csv',
 'ATLAS_8/ATLAS8-04y08.csv',
 'ATLAS_8/ATLAS8-08y12.csv',
 'ATLAS_8/ATLAS8-116Q150.csv',
 'ATLAS_8/ATLAS8-12y16.csv',
 'ATLAS_8/ATLAS8-16y20.csv',
 'ATLAS_8/ATLAS8-20y24.csv',
 'ATLAS_8/ATLAS8-46Q66.csv',
 'CDF_I/CDF1.csv',
 'CDF_II/CDF2.csv',
 'CMS_7/CMS7.csv',
 'CMS_8/CMS8.csv',
 'CMS_13/CMS13-00y04.csv',
 'CMS_13/CMS13-04y08.csv',
 'CMS_13/CMS13-08y12.csv',
 'CMS_13/CMS13-106Q170.csv',
 'CMS_13/CMS13-12y16.csv',
 'CMS_13/CMS13-16y24.csv',
 'CMS_13/CMS13-170Q350.csv',
 'CMS_13/CMS13-350Q1000.csv',
 'D0_I/D01.csv',
 'D0_II/D02.csv',
 'D0_II_mu/D02mu.csv',
 'E288/E228-200-4Q5.csv',
 'E288/E228-200-5Q6.csv',
 'E288/E228-200-6Q7.csv',
 'E288/E228-200-7Q8.csv',
 'E288/E228-200-8Q9.csv',
 'E288/E228-300-11Q12.csv',
 'E288/E228-300-4Q5.csv',
 'E288/E228-300-5Q6.csv',
 'E288/E228-300-6Q7.csv',
 'E288/E228-300-7Q8.csv',
 'E288/E228-300-8Q9.csv',
 'E288/E228-400-11Q12.csv',
 'E28

Generate Tables

In [8]:
if if_generate_table:
    for exp in experiments:
        exp_dir = Path(table_root) / exp
        if exp_dir.exists():
            shutil.rmtree(exp_dir) 
        exp_dir.mkdir(parents=True, exist_ok=True)

In [9]:
data_list = []
path_list = []
df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

for file in file_names:

    df = to_float64(pd.read_csv(f"{file_root}/{file}"))
    df_prediction = pd.DataFrame([])

    name_short = Path(file).stem
    if name_short in total_xsec_names:
        total_xsec = df_total_xsec[df_total_xsec['name'] == name_short]["total_xsec"].values[0]    
        print(f"{name_short}'s total xsec = {total_xsec} added")
        
    for i in range(len(df)):

        data = df.iloc[i].to_dict()
        if name_short in total_xsec_names:
            data['total_xsec'] = total_xsec

        base = os.path.splitext(file)[0]                 
        table_path = os.path.join(table_root, f"{base}/{i}.jls")
        os.makedirs(os.path.dirname(table_path), exist_ok=True)

        data_list.append(data)
        path_list.append(table_path)

ATLAS7-00y10's total xsec = 1.0 added
ATLAS7-10y20's total xsec = 1.0 added
ATLAS7-20y24's total xsec = 1.0 added
CMS7's total xsec = 1.0 added
CMS8's total xsec = 1.0 added
D02's total xsec = 1.0 added
D02mu's total xsec = 1.0 added


Calculate

In [10]:
if if_generate_table:
    Main.DY_table_generator(data_list, path_list)
    print("tables generated")
else:
    predictions = Main.DY_integrated_xsec_pmap(data_list, path_list)
    
    import pickle
    os.makedirs(f"../Predict/{Main.table_name}", exist_ok=True)
    store_path = f"../Predict/{Main.table_name}/{Main.i_member}.pkl"
    if os.path.exists(store_path):
        os.remove(store_path)
    with open(store_path, "wb") as f:
        pickle.dump(predictions, f)
    #display(predictions)
    print("predictions saved")

predictions saved
